# Nuage de points sur carte : éditions de Venise, Paris et Lyon

Pour ces trois villes d'édition des *Métamorphoses* d'Ovide, un nuage de points sur une
**vraie carte géographique historique** (même fond `OpenHistoricalMap` que
`01_carte_circulation.ipynb`) plutôt qu'un graphique abstrait : Venise, Paris et Lyon sont à leur
position réelle, et autour de chaque ville un petit nuage de points en spirale, un point par
**éditeur** ayant publié dans cette ville (taille ∝ nombre d'éditions publiées). À droite de
chaque carte, une frise chronologique reprend le détail édition par édition, et un tableau
dépliable liste tout, éditeur compris.

Même source de données que `01_carte_circulation.ipynb` : `retours_celine/BNU_corpus.ods`
(feuille `Synthèse`).

In [1]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "nuage_editions_venise_paris_lyon.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS cellule par cellule que pour la carte (`pandas.read_excel` masque des
colonnes de ce fichier — voir `01_carte_circulation.ipynb` pour le détail). On ne garde que les
éditions dont la ville normalisée est Lyon, Paris ou Venise.

In [2]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence (même logique que
    01_carte_circulation.ipynb) : certaines cellules contiennent plusieurs liens séparés par
    ';' (ex. plusieurs notices catalogue) — on ne garde que le premier, sinon le lien final
    serait une concaténation invalide (ex. Lyon 1516, Biblioteca Digital Ovidiana contenait
    une seule URL mais suivie d'un ';?%3E' résiduel qui aurait été inclus tel quel)."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def graveur_ou_inconnu(g):
    """Repli neutre pour un graveur non identifié ('?', 'inaccessible', case vide) — les
    'AnonymeXXXX' (X = année) sont eux déjà des identifiants distincts d'un graveur à l'autre,
    contrairement à 's.n.' pour les éditeurs (voir plus bas) : pas besoin de les numéroter."""
    g = (g or "").strip()
    if not g or g.lower() in {"?", "inaccessible"}:
        return "Graveur non identifié"
    return g

# Seules les 3 villes qui nous intéressent ici (pas besoin de la table de correspondance
# complète de 01_carte_circulation.ipynb, qui couvre 19 villes)
VILLES_RETENUES = {"Paris": "Paris", "[Paris]": "Paris", "Lyon": "Lyon", "Venise": "Venise"}
ORDRE_VILLES = ["Lyon", "Paris", "Venise"]

def categorie_technique(t):
    t = (t or "").strip().lower()
    if t == "bois":
        return "bois"
    if t == "cuivre":
        return "cuivre"
    return "inconnue"  # vide, "inaccessible", "?", etc. -> repli neutre plutôt que fragmenter

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

editions = []
for row in corpus:
    ville = VILLES_RETENUES.get(row.get("ville", "").strip())
    if ville is None:
        continue
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    editions.append({
        "ville": ville,
        "annee": annee,
        "titre": titre.strip(),
        "technique": categorie_technique(row.get("technique", "")),
        "graveur": graveur_ou_inconnu(row.get(COL_GRAVEUR, "")),
        "publisher": row.get("publisher", "").strip() or "Éditeur non identifié",
        "lien": extraire_lien(row),
    })

# "s.n." ("sine nomine" : éditeur non mentionné sur l'édition) n'est pas un nom d'éditeur —
# plusieurs éditions "s.n." dans une même ville ne sont pas forcément du même éditeur. Sans
# ce correctif, elles seraient regroupées à tort en un seul point sur la carte (comme si
# c'était un unique éditeur très actif). On les distingue donc par un numéro (s.n.1, s.n.2...),
# uniquement quand il y en a plus d'une dans la ville (sinon le numéro n'apporte rien).
for ville in ORDRE_VILLES:
    inconnus = [e for e in editions if e["ville"] == ville and e["publisher"].strip().lower() == "s.n."]
    if len(inconnus) > 1:
        for i, e in enumerate(sorted(inconnus, key=lambda e: e["annee"]), start=1):
            e["publisher"] = f"s.n.{i}"

print(len(editions), "éditions retenues (Lyon, Paris, Venise)")
for v in ORDRE_VILLES:
    sous = [e for e in editions if e["ville"] == v]
    print(f"  {v:8s} {len(sous):2d} éditions, {min(e['annee'] for e in sous)}–{max(e['annee'] for e in sous)}, "
          f"{len({e['publisher'] for e in sous})} éditeurs distincts, "
          f"{len({e['graveur'] for e in sous})} graveurs distincts")

from collections import Counter
print("techniques :", Counter(e["technique"] for e in editions))

64 éditions retenues (Lyon, Paris, Venise)
  Lyon     21 éditions, 1510–1697, 14 éditeurs distincts, 8 graveurs distincts
  Paris    25 éditions, 1493–1737, 19 éditeurs distincts, 13 graveurs distincts
  Venise   18 éditions, 1497–1624, 14 éditeurs distincts, 10 graveurs distincts
techniques : Counter({'bois': 43, 'cuivre': 18, 'inconnue': 3})


## 2. Choix de visualisation

- **Forme** : les deux à la fois, combinées plutôt que l'une remplaçant l'autre. Pour chaque
  ville, un panneau avec (a) une **carte géographique historique** (position vraie, nuage en
  spirale de phyllotaxie autour) et (b) juste en dessous une **frise chronologique** compacte
  (axe = année 1490-1750) pour la même ville — la carte donne le lieu, la frise donne le
  moment, les deux se lisent ensemble sans se remplacer.
- **Granularité** : la carte et la frise ne montrent pas la même chose. Sur la **carte**, un
  point = un **éditeur** actif dans cette ville, avec une taille proportionnelle à son nombre
  d'éditions (aire ∝ nombre d'éditions, donc rayon ∝ racine — même convention que les villes
  de `01_carte_circulation.ipynb`) : ça évite un nuage de points tous identiques et fait
  ressortir les éditeurs les plus actifs. Sur la **frise**, un point = une **édition**
  individuelle, positionnée par année : le détail édition par édition reste consultable, sans
  être noyé sur la carte.
- **Fond de carte** : `OpenHistoricalMap` (frontières et toponymes d'époque), le même fond
  que `01_carte_circulation.ipynb`, plutôt qu'un fond de carte contemporain.
- **Couleur** : encode l'**éditeur**, pas la technique de gravure. La technique bois/cuivre
  n'est pas une catégorie pertinente pour cette carte (elle reste indiquée en texte dans les
  infobulles, popups et le tableau détaillé, simplement plus en couleur). Chaque éditeur d'une
  ville reçoit une teinte distincte par rotation à l'angle d'or sur le cercle chromatique —
  pas de palette manuelle à maintenir, ni de collision même avec beaucoup d'éditeurs (jusqu'à
  19 à Paris). Un point de frise reprend la couleur de son éditeur, pour le relier visuellement
  à sa bulle sur la carte.
- **Interaction** : survol = infobulle rapide (aperçu). **Clic** = fiche détaillée qui reste
  affichée tant qu'on ne clique pas ailleurs — sur la carte via une popup Leaflet listant
  chaque édition de l'éditeur avec son lien "voir" ; sur la frise via l'infobulle personnalisée
  qui s'épingle. Un simple survol ne suffisait pas pour atteindre le lien "voir" (il disparaît
  dès que la souris quitte le petit point pour aller cliquer dessus) — d'où ce passage en deux
  temps. Vue tableau dépliable (accessibilité), colonne éditeur comprise.
- **Mode sombre** : mêmes couleurs de fond/texte validées séparément que la version
  précédente ; les couleurs par éditeur, elles, sont calculées une fois et ne changent pas
  avec le thème (contour foncé/clair du point pour rester lisible dans les deux cas).

In [3]:
import math
from collections import Counter

LIBELLES_TECHNIQUE = {"bois": "Bois", "cuivre": "Cuivre", "inconnue": "Technique inconnue"}

# Mêmes coordonnées que 01_carte_circulation.ipynb
VILLES_COORDS = {"Lyon": (45.764, 4.8357), "Paris": (48.8566, 2.3522), "Venise": (45.4408, 12.3155)}

RAYON_MAX_KM = 22            # étalement maximal du nuage d'éditeurs autour de chaque ville
ANGLE_OR = math.radians(137.508)  # angle d'or : répartition régulière en spirale, sans grille

def decalage_spirale(i, n, rayon_max_km):
    """Spirale de phyllotaxie (rayon ∝ √i, angle = i × angle d'or) : répartit n'importe quel
    nombre de points régulièrement autour d'un centre, sans qu'ils se chevauchent."""
    if n <= 1:
        return 0.0, 0.0
    rayon = rayon_max_km * math.sqrt(i / (n - 1))
    angle = i * ANGLE_OR
    return rayon * math.cos(angle), rayon * math.sin(angle)

def deplacer_latlon(lat, lon, dx_km, dy_km):
    """Déplace un point de (dx_km vers l'est, dy_km vers le nord) en degrés lat/lon."""
    dlat = dy_km / 111.0
    dlon = dx_km / (111.0 * math.cos(math.radians(lat)))
    return lat + dlat, lon + dlon

# --- Frise chronologique compacte (une par ville, à droite de sa carte) ---
# Plus haute que la première version : les éditions proches dans le temps ont besoin de
# place pour être clairement séparées verticalement (voir plus bas).
FRISE_LARGEUR, FRISE_HAUTEUR = 320, 130
FRISE_MARGE = {"gauche": 12, "droite": 16, "haut": 18, "bas": 26}
FRISE_ANNEE_MIN, FRISE_ANNEE_MAX = 1490, 1750
FRISE_HAUTEUR_PLOT = FRISE_HAUTEUR - FRISE_MARGE["haut"] - FRISE_MARGE["bas"]
FRISE_Y_CENTRE = FRISE_MARGE["haut"] + FRISE_HAUTEUR_PLOT / 2

def frise_x(annee):
    t = (annee - FRISE_ANNEE_MIN) / (FRISE_ANNEE_MAX - FRISE_ANNEE_MIN)
    return FRISE_MARGE["gauche"] + t * (FRISE_LARGEUR - FRISE_MARGE["gauche"] - FRISE_MARGE["droite"])

# Deux points dans la même case de cette largeur (en pixels) sont jugés "trop proches
# horizontalement" et regroupés pour l'étalement vertical. Des cases de largeur fixe
# plutôt qu'un chaînage de proche en proche : le chaînage a tendance à faire boule de neige
# (si A est proche de B et B proche de C, A et C finissent regroupés même s'ils sont loin
# l'un de l'autre), ce qui étalait presque toutes les éditions d'une ville sur toute la
# hauteur au lieu de ne séparer que les vraies coïncidences locales.
LARGEUR_CASE_PX = 16

groupes = {}
for e in editions:
    groupes.setdefault(e["ville"], []).append(e)

# --- Points de la frise : un par édition (positionnement géographique séparé, voir plus bas
# pour les points de la carte qui eux sont groupés par éditeur) ---
points = []
for ville, groupe in groupes.items():
    groupe_trie = sorted(groupe, key=lambda e: e["annee"])
    n = len(groupe_trie)

    # Regroupe par case fixe le long de l'axe des années (pas par année exacte, ni par
    # chaînage de proche en proche)
    cases = {}
    for idx, e in enumerate(groupe_trie):
        x = frise_x(e["annee"])
        case = int((x - FRISE_MARGE["gauche"]) // LARGEUR_CASE_PX)
        cases.setdefault(case, []).append(idx)
    position_dans_case = {}
    for indices in cases.values():
        for j, idx in enumerate(indices):
            position_dans_case[idx] = (j, len(indices))

    for i, e in enumerate(groupe_trie):
        # Position sur la frise chronologique : les éditions tombant dans la même case
        # (même année ou années très voisines) sont nettement séparées verticalement
        # (jusqu'à 25px, largement plus que le rayon des points)
        j, m = position_dans_case[i]
        espacement_f = min(25, (FRISE_HAUTEUR_PLOT * 0.85) / max(m - 1, 1)) if m > 1 else 0
        fy = FRISE_Y_CENTRE + (j - (m - 1) / 2) * espacement_f
        fx = frise_x(e["annee"])

        points.append({
            "fx": round(fx, 1),
            "fy": round(fy, 1),
            "ville": ville,
            "annee": e["annee"],
            "titre": e["titre"],
            "technique": e["technique"],
            "graveur": e["graveur"],
            "editeur": e["publisher"],
            "lien": e["lien"],
        })

print(len(points), "points positionnés sur la frise (un par édition)")

64 points positionnés sur la frise (un par édition)


**Points de la carte** : contrairement à la frise, un point ici représente un **éditeur**
(regroupement de toutes ses éditions dans cette ville), positionné en spirale comme
précédemment mais avec `n` = nombre d'éditeurs de la ville plutôt que nombre d'éditions.
L'éditeur le plus actif est placé au centre (i=0 dans la spirale).

**Couleur** : un point = un éditeur, avec une couleur qui lui est propre (la technique de
gravure bois/cuivre n'est pas une catégorie pertinente pour cette carte-ci — elle reste
indiquée en texte dans les infobulles et le tableau, mais n'est plus encodée par couleur). Les
teintes sont réparties par rotation à l'angle d'or (même principe que la spirale ci-dessus,
appliqué cette fois aux teintes plutôt qu'aux positions) : chaque éditeur d'une ville a une
couleur distincte, sans configuration manuelle même si leur nombre change. Les points de la
frise reprennent la couleur de leur éditeur, pour relier visuellement un point de frise à sa
bulle sur la carte.

**Bascule éditeur / graveur** : la cellule suivante calcule le même regroupement, mais par
**graveur** plutôt que par éditeur (`points_graveurs`, même logique de spirale et de couleur).
La carte HTML propose un bouton pour basculer de l'un à l'autre — les deux jeux de bulles sont
calculés à l'avance ici, la bascule ne fait que changer laquelle est affichée (section 4).

In [4]:
import colorsys

def couleur_categorielle(i, saturation=0.55, luminosite=0.48):
    """Couleur hex distincte pour l'index i, par rotation à l'angle d'or (voir ANGLE_OR
    plus haut) : n'importe quel nombre de catégories reste bien réparti sur le cercle
    chromatique, sans avoir à choisir une palette manuelle à l'avance."""
    teinte = (i * 137.508 % 360) / 360
    r, g, b = colorsys.hls_to_rgb(teinte, luminosite, saturation)
    return "#{:02x}{:02x}{:02x}".format(round(r * 255), round(g * 255), round(b * 255))

groupes_editeurs = {}
for e in editions:
    groupes_editeurs.setdefault((e["ville"], e["publisher"]), []).append(e)

points_editeurs = []
couleurs_editeurs = {}  # (ville, editeur) -> couleur hex, réutilisée pour les points de la frise
for ville in VILLES_COORDS:
    editeurs_ville = sorted(
        {cle[1] for cle in groupes_editeurs if cle[0] == ville},
        key=lambda editeur: (-len(groupes_editeurs[(ville, editeur)]), editeur)
    )
    n = len(editeurs_ville)
    lat0, lon0 = VILLES_COORDS[ville]
    for i, editeur in enumerate(editeurs_ville):
        couleur = couleur_categorielle(i)
        couleurs_editeurs[(ville, editeur)] = couleur
        eds = sorted(groupes_editeurs[(ville, editeur)], key=lambda e: e["annee"])
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)
        points_editeurs.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "ville": ville,
            "editeur": editeur,
            "couleur": couleur,
            "editions": [
                {"annee": e["annee"], "titre": e["titre"], "technique": e["technique"], "lien": e["lien"]}
                for e in eds
            ],
        })

# Les points de la frise (un par édition) reprennent la couleur de leur éditeur, pour relier
# visuellement un point de frise à sa bulle d'éditeur sur la carte.
for p in points:
    p["couleurEditeur"] = couleurs_editeurs[(p["ville"], p["editeur"])]

print(len(points_editeurs), "éditeurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)")
for ville in VILLES_COORDS:
    sous = [p for p in points_editeurs if p["ville"] == ville]
    plus_actif = max(sous, key=lambda p: len(p["editions"]))
    print(f"  {ville:8s} {len(sous):2d} éditeurs — le plus actif : {plus_actif['editeur']} "
          f"({len(plus_actif['editions'])} éditions)")

47 éditeurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)
  Lyon     14 éditeurs — le plus actif : Jean de Tournes (5 éditions)
  Paris    19 éditeurs — le plus actif : Henri de Marnef et Guillaume Cavellat (5 éditions)
  Venise   14 éditeurs — le plus actif : Francesco de' Franceschi (4 éditions)


In [5]:
# Même logique que points_editeurs, mais groupée par graveur plutôt que par éditeur.
groupes_graveurs = {}
for e in editions:
    groupes_graveurs.setdefault((e["ville"], e["graveur"]), []).append(e)

points_graveurs = []
couleurs_graveurs = {}
for ville in VILLES_COORDS:
    graveurs_ville = sorted(
        {cle[1] for cle in groupes_graveurs if cle[0] == ville},
        key=lambda graveur: (-len(groupes_graveurs[(ville, graveur)]), graveur)
    )
    n = len(graveurs_ville)
    lat0, lon0 = VILLES_COORDS[ville]
    for i, graveur in enumerate(graveurs_ville):
        couleur = couleur_categorielle(i)
        couleurs_graveurs[(ville, graveur)] = couleur
        eds = sorted(groupes_graveurs[(ville, graveur)], key=lambda e: e["annee"])
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)
        points_graveurs.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "ville": ville,
            "graveur": graveur,
            "couleur": couleur,
            "editions": [
                {"annee": e["annee"], "titre": e["titre"], "technique": e["technique"], "lien": e["lien"]}
                for e in eds
            ],
        })

# Les points de la frise reprennent aussi la couleur de leur graveur, utilisée quand la
# bascule de la carte est sur "Graveur" (section 4).
for p in points:
    p["couleurGraveur"] = couleurs_graveurs[(p["ville"], p["graveur"])]

print(len(points_graveurs), "graveurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)")
for ville in VILLES_COORDS:
    sous = [p for p in points_graveurs if p["ville"] == ville]
    plus_actif = max(sous, key=lambda p: len(p["editions"]))
    print(f"  {ville:8s} {len(sous):2d} graveurs — le plus actif : {plus_actif['graveur']} "
          f"({len(plus_actif['editions'])} éditions)")

31 graveurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)
  Lyon      8 graveurs — le plus actif : Leroy II, Guillaume (5 éditions)
  Paris    13 graveurs — le plus actif : Salomon, Bernard (5 éditions)
  Venise   10 graveurs — le plus actif : Anonyme1572 (5 éditions)


## 3. Génération de la carte + frise (HTML autonome)

Mise en page en **deux colonnes** : à gauche, les trois cartes empilées verticalement (une
par ville) ; à droite, les trois frises chronologiques empilées, chacune alignée sur la même
ligne que sa carte.

In [6]:
def ligne_tableau(p):
    lien_html = f'<a href="{p["lien"]}" target="_blank">voir</a>' if p["lien"] else ""
    return (
        f'<tr><td>{p["annee"]}</td><td>{p["titre"]}</td>'
        f'<td>{p["editeur"]}</td>'
        f'<td>{LIBELLES_TECHNIQUE[p["technique"]]}</td><td>{p["graveur"]}</td>'
        f'<td>{lien_html}</td></tr>'
    )

def tableau_ville(ville):
    lignes = "\n".join(
        ligne_tableau(p) for p in sorted(points, key=lambda p: p["annee"]) if p["ville"] == ville
    )
    return f'''<table class="tableau-detaille" id="tableau-{ville}">
    <thead><tr><th>Année</th><th>Titre</th><th>Éditeur</th><th>Technique</th><th>Graveur</th><th>Lien</th></tr></thead>
    <tbody>
      {lignes}
    </tbody>
  </table>'''

# --- Frise chronologique : axe (partagé, même échelle pour les 3 villes) + points par ville ---
FRISE_TICKS = [1500, 1600, 1700]

def frise_axes_svg():
    x_debut, x_fin = frise_x(FRISE_ANNEE_MIN), frise_x(FRISE_ANNEE_MAX)
    elements = [
        # ligne de base : l'axe du temps lui-même, du début à la fin de la période
        f'<line x1="{x_debut:.1f}" y1="{FRISE_Y_CENTRE:.1f}" x2="{x_fin:.1f}" y2="{FRISE_Y_CENTRE:.1f}" class="ligne-base"/>',
        # petite pointe de flèche à droite : le sens du temps
        f'<path d="M {x_fin - 1:.1f} {FRISE_Y_CENTRE - 4:.1f} L {x_fin + 6:.1f} {FRISE_Y_CENTRE:.1f} '
        f'L {x_fin - 1:.1f} {FRISE_Y_CENTRE + 4:.1f} Z" class="pointe-base"/>',
    ]
    for t in FRISE_TICKS:
        x = frise_x(t)
        elements.append(
            f'<line x1="{x:.1f}" y1="{FRISE_MARGE["haut"]}" x2="{x:.1f}" '
            f'y2="{FRISE_MARGE["haut"] + FRISE_HAUTEUR_PLOT:.1f}" class="grille-frise"/>'
        )
        elements.append(f'<line x1="{x:.1f}" y1="{FRISE_Y_CENTRE-4:.1f}" x2="{x:.1f}" y2="{FRISE_Y_CENTRE+4:.1f}" class="tick-frise"/>')
        elements.append(
            f'<text x="{x:.1f}" y="{FRISE_MARGE["haut"] + FRISE_HAUTEUR_PLOT + 16:.1f}" '
            f'text-anchor="middle" class="etiquette-annee-frise">{t}</text>'
        )
    return "\n".join(elements)

FRISE_AXES = frise_axes_svg()

def frise_cercles_ville(ville):
    # Couleur par éditeur au chargement (mode par défaut de la bascule) — mise à jour en JS
    # (couleurEditeur / couleurGraveur) quand on bascule.
    cercles = []
    for i, p in enumerate(points):
        if p["ville"] != ville:
            continue
        cercles.append(
            f'<circle class="point-frise" data-i="{i}" cx="{p["fx"]}" cy="{p["fy"]}" r="5" fill="{p["couleurEditeur"]}"/>'
        )
    return "\n".join(cercles)

# Deux colonnes (carte à gauche, frise à droite) : chaque ville ajoute au flux, dans l'ordre,
# [carte, frise, bouton (pleine largeur), tableau (pleine largeur)] ; avec 2 colonnes de
# grille, le bouton et le tableau — qui s'étendent sur les 2 colonnes — retombent
# naturellement sur leur propre ligne juste sous la paire carte/frise de leur ville.
cellules = []
for ville in VILLES_COORDS:
    cellules.append(f'<div class="cellule-carte"><h2>{ville}</h2><div class="carte" id="carte-{ville}"></div></div>')
    cellules.append(
        f'<div class="panneau-frise"><svg class="frise" viewBox="0 0 {FRISE_LARGEUR} {FRISE_HAUTEUR}">\n'
        f'    {FRISE_AXES}\n'
        f'    {frise_cercles_ville(ville)}\n'
        f'  </svg></div>'
    )
    cellules.append(f'<button class="bascule action-ville" data-ville="{ville}">Afficher le tableau détaillé</button>')
    cellules.append(tableau_ville(ville))
blocs_carte = "\n".join(cellules)

TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Nuage de points : éditions de Venise, Paris et Lyon</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet"/>
<script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
<script src="https://unpkg.com/@maplibre/maplibre-gl-leaflet@0.0.20/leaflet-maplibre-gl.js"></script>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1100px; margin:0 auto; padding:16px 20px 32px; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 10px; }

  .commandes-globales { display:flex; align-items:center; gap:8px; font-size:12px;
    color:var(--texte-att); margin:0 0 14px; }
  .bascule-groupement { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  .bascule-groupement.actif { background:var(--texte-fort); color:var(--surface); border-color:var(--texte-fort); }

  /* Colonne de gauche = cartes, colonne de droite = frises ; le bouton et le tableau de
     chaque ville s'étendent sur les 2 colonnes, juste sous sa paire carte/frise. */
  .rangee-cartes { display:grid; grid-template-columns:3fr 2fr; align-items:center;
    gap:8px 20px; margin-top:14px; }
  @media (max-width:760px) { .rangee-cartes { grid-template-columns:1fr; } }
  .cellule-carte h2 { font-size:15px; margin:0 0 6px; color:var(--texte-fort); }
  .carte { height:300px; border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.25); }

  .panneau-frise { background:var(--surface); border:1px solid var(--trait); border-radius:6px;
    box-shadow:0 1px 6px rgba(0,0,0,.15); padding:6px 6px 2px; }
  .frise { width:100%; height:auto; overflow:visible; display:block; }
  .ligne-base { stroke:var(--texte-att); stroke-width:1.5; }
  .pointe-base { fill:var(--texte-att); }
  .grille-frise { stroke:var(--trait); stroke-width:1; }
  .tick-frise { stroke:var(--texte-att); stroke-width:1; }
  .etiquette-annee-frise { font-size:10px; fill:var(--texte-att); }
  .point-frise { stroke:var(--contour-point); stroke-width:1.2; fill-opacity:.9; cursor:pointer;
    transition:r .15s, fill .2s; filter:drop-shadow(0 1px 1px rgba(0,0,0,.35)); }
  .point-frise:hover, .point-frise.actif { r:8; fill-opacity:1; }

  .action-ville { grid-column:1 / -1; justify-self:start; margin:2px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { grid-column:1 / -1; width:100%; border-collapse:collapse;
    font-size:12px; margin:2px 0 6px; display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-lien); }

  .leaflet-tooltip { font-family:Georgia,serif; font-size:12px; background:var(--surface);
    border:1px solid var(--contour-point); color:var(--texte-fort); padding:4px 9px;
    box-shadow:0 1px 5px rgba(0,0,0,.3); }
  /* Popups Leaflet (clic sur un éditeur ou un graveur) : même thème parchemin, contenu
     défilable si la liste d'éditions est longue, liens "voir" bien cliquables (contrairement
     à l'infobulle au survol, une popup reste ouverte tant qu'on ne clique pas ailleurs). */
  .leaflet-popup-content-wrapper { background:var(--surface); color:var(--texte-fort);
    border:1px solid var(--contour-point); border-radius:6px; }
  .leaflet-popup-tip { background:var(--surface); border:1px solid var(--contour-point); }
  .leaflet-popup-content { font-family:Georgia,serif; font-size:12px; max-height:220px;
    overflow-y:auto; margin:10px 12px; }
  .leaflet-popup-content hr { border:none; border-top:1px solid var(--trait); margin:4px 0; }
  .ed-popup { padding:3px 0; border-bottom:1px solid var(--trait); }
  .ed-popup:last-child { border-bottom:none; }
  .ed-popup a { color:var(--c-lien); }

  /* Infobulle personnalisée de la frise : au survol elle suit la souris (aperçu rapide, SANS
     lien — voir contenuApercu) ; au clic elle s'épingle sur place (pointer-events actifs, un
     bouton "×" la referme) et affiche alors le lien "voir", cliquable puisque l'infobulle ne
     bouge plus et accepte les événements de souris. */
  .infobulle { position:absolute; pointer-events:none; background:var(--surface);
    border:1px solid var(--contour-point); border-radius:5px; padding:6px 10px; font-size:12px;
    max-width:260px; opacity:0; transition:opacity .1s; box-shadow:0 2px 8px rgba(0,0,0,.3); z-index:2000; }
  .infobulle.epinglee { pointer-events:auto; }
  .infobulle a { color:var(--c-lien); }
  .infobulle .fermer-infobulle { position:absolute; top:2px; right:6px; cursor:pointer;
    color:var(--texte-att); font-size:13px; }
</style></head><body>
<div class="page">
  <h1>Éditions de Venise, Paris et Lyon</h1>
  <p class="souschapo">Sur la carte, un point regroupe plusieurs éditions (couleur propre,
    taille proportionnelle au nombre d'éditions) ; sur la frise, un point = une édition
    individuelle, positionnée par année et colorée comme son groupe sur la carte. Cliquer un
    point affiche le détail avec le lien "voir" (le survol seul n'affiche qu'un aperçu).</p>
  <div class="commandes-globales">
    <span>Regrouper les points de la carte par :</span>
    <button class="bascule-groupement actif" data-mode="editeur">Éditeur</button>
    <button class="bascule-groupement" data-mode="graveur">Graveur</button>
  </div>
  <div class="rangee-cartes">
    __BLOCS_CARTE__
  </div>
  <div class="infobulle" id="infobulle"></div>
</div>
<script>
  const points = __POINTS__;
  const pointsEditeurs = __POINTS_EDITEURS__;
  const pointsGraveurs = __POINTS_GRAVEURS__;
  const villesCoords = __VILLES_COORDS__;
  const libellesTechnique = __LIBELLES__;

  // Deux variantes du contenu : l'aperçu au survol n'inclut PAS le lien (comme dans
  // 01_carte_circulation.ipynb, où le survol d'une flèche n'affiche jamais de lien cliquable) ;
  // seule la version détaillée, affichée une fois l'infobulle épinglée par un clic (donc
  // vraiment cliquable, cf. .infobulle.epinglee plus haut), inclut le lien "voir". Montrer le
  // lien dès le survol serait trompeur : il aurait l'air cliquable sans l'être, puisque
  // l'infobulle de survol ignore les clics tant qu'elle n'est pas épinglée.
  function contenuApercu(p) {
    return '<b>' + p.titre + '</b><br>' +
      '<span>' + p.ville + ', ' + p.annee + ' · ' + libellesTechnique[p.technique] + '</span>' +
      (p.graveur ? '<br>' + p.graveur : '') +
      (p.editeur ? '<br><i>' + p.editeur + '</i>' : '');
  }
  function contenuDetaille(p) {
    const lien = p.lien ? '<br><a href="' + p.lien + '" target="_blank">→ voir</a>' : '';
    return contenuApercu(p) + lien;
  }

  // Construit les marqueurs Leaflet d'un jeu de points (éditeurs ou graveurs) pour une ville.
  // `cle` vaut 'editeur' ou 'graveur' : nom de la clé qui porte le nom du groupe dans `p`.
  function construireMarqueurs(map, liste, ville, cle) {
    return liste.filter(p => p.ville === ville).map(p => {
      const n = p.editions.length;
      const rayon = 6 + Math.sqrt(n) * 5;
      const nom = p[cle];
      const editionsTriees = p.editions.slice().sort((a, b) => a.annee - b.annee);
      const listeEditions = editionsTriees.map(e =>
        '<div class="ed-popup"><b>' + e.annee + '</b> — ' + e.titre +
        ' <i>(' + libellesTechnique[e.technique] + ')</i>' +
        (e.lien ? ' <a href="' + e.lien + '" target="_blank">→ voir</a>' : '') + '</div>'
      ).join('');
      return L.circleMarker([p.lat, p.lon], {
        radius: rayon, color:'#3e2c23', weight:1, fillColor: p.couleur, fillOpacity:.85
      })
        // survol : aperçu rapide (nom + nombre d'éditions), pas de lien — cohérent avec la
        // frise et avec 01_carte_circulation.ipynb (le survol ne montre jamais de lien cliquable)
        .bindTooltip('<b>' + nom + '</b><br>' + ville + ' · ' + n + ' édition(s)', {sticky:true, maxWidth:220})
        // clic : popup Leaflet, nativement interactive, avec le détail de chaque édition et
        // son lien "voir"
        .bindPopup('<b>' + nom + '</b><br>' + ville + ', ' + n + ' édition(s)<hr>' + listeEditions, {maxWidth:280});
    });
  }

  // Une carte Leaflet par ville, chacune cadrée sur son propre nuage (pas sur les 3 villes à
  // la fois : Lyon/Paris/Venise sont loin les unes des autres, une seule carte les englobant
  // montrerait surtout du vide entre elles). Fond OpenHistoricalMap (frontières et toponymes
  // d'époque), comme dans 01_carte_circulation.ipynb. Les deux jeux de marqueurs (éditeur et
  // graveur) sont construits à l'avance dans des L.layerGroup séparés : basculer ne fait que
  // retirer l'un et ajouter l'autre, sans reconstruire quoi que ce soit.
  const calquesParVille = {};
  for (const ville of Object.keys(villesCoords)) {
    const map = L.map('carte-' + ville, {scrollWheelZoom:false});
    L.maplibreGL({
      style: 'https://www.openhistoricalmap.org/map-styles/main/main.json',
      attribution: '© OpenHistoricalMap contributors'
    }).addTo(map);

    const editeursVille = pointsEditeurs.filter(p => p.ville === ville);
    const bornes = L.latLngBounds(editeursVille.map(p => [p.lat, p.lon]));
    map.fitBounds(bornes, {padding:[36, 36]});

    const calqueEditeur = L.layerGroup(construireMarqueurs(map, pointsEditeurs, ville, 'editeur'));
    const calqueGraveur = L.layerGroup(construireMarqueurs(map, pointsGraveurs, ville, 'graveur'));
    calqueEditeur.addTo(map);
    calquesParVille[ville] = {map, editeur: calqueEditeur, graveur: calqueGraveur};
  }

  // Bascule globale "Éditeur" / "Graveur" : change le calque affiché sur les 3 cartes, et la
  // couleur des points de la frise (pour rester cohérente avec ce qui est groupé sur la carte).
  let modeGroupement = 'editeur';
  document.querySelectorAll('.bascule-groupement').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const mode = bouton.dataset.mode;
      if (mode === modeGroupement) return;
      document.querySelectorAll('.bascule-groupement').forEach(b => b.classList.toggle('actif', b === bouton));
      for (const ville of Object.keys(calquesParVille)) {
        const c = calquesParVille[ville];
        c.map.removeLayer(c[modeGroupement]);
        c.map.addLayer(c[mode]);
      }
      document.querySelectorAll('.point-frise').forEach(cercle => {
        const p = points[+cercle.dataset.i];
        cercle.setAttribute('fill', mode === 'editeur' ? p.couleurEditeur : p.couleurGraveur);
      });
      modeGroupement = mode;
    });
  });

  // --- Infobulle partagée pour les points de la frise chronologique (SVG, à droite de
  // chaque carte) : survol = aperçu qui suit la souris (contenuApercu, sans lien), clic =
  // épinglée sur place (contenuDetaille, avec lien, pointer-events actifs). ---
  const infobulle = document.getElementById('infobulle');
  const page = document.querySelector('.page');
  let infobulleEpinglee = false;

  function positionnerInfobulle(ev) {
    const r = page.getBoundingClientRect();
    infobulle.style.left = (ev.clientX - r.left + 14) + 'px';
    infobulle.style.top = (ev.clientY - r.top + 14) + 'px';
  }
  function fermerInfobulle() {
    infobulleEpinglee = false;
    infobulle.classList.remove('epinglee');
    infobulle.style.opacity = 0;
    document.querySelectorAll('.point-frise.actif').forEach(c => c.classList.remove('actif'));
  }

  document.querySelectorAll('.point-frise').forEach(cercle => {
    const p = points[+cercle.dataset.i];
    cercle.addEventListener('mouseenter', () => {
      if (infobulleEpinglee) return;
      cercle.classList.add('actif');
      infobulle.innerHTML = contenuApercu(p);
      infobulle.style.opacity = 1;
    });
    cercle.addEventListener('mousemove', (ev) => {
      if (infobulleEpinglee) return;
      positionnerInfobulle(ev);
    });
    cercle.addEventListener('mouseleave', () => {
      if (infobulleEpinglee) return;
      cercle.classList.remove('actif');
      infobulle.style.opacity = 0;
    });
    cercle.addEventListener('click', (ev) => {
      ev.stopPropagation();
      positionnerInfobulle(ev);
      infobulle.innerHTML = contenuDetaille(p) + '<span class="fermer-infobulle" title="Fermer">×</span>';
      infobulle.style.opacity = 1;
      infobulle.classList.add('epinglee');
      infobulleEpinglee = true;
      document.querySelectorAll('.point-frise.actif').forEach(c => c.classList.remove('actif'));
      cercle.classList.add('actif');
      infobulle.querySelector('.fermer-infobulle').addEventListener('click', fermerInfobulle);
    });
  });

  // Cliquer ailleurs sur la page referme l'infobulle épinglée
  document.addEventListener('click', (ev) => {
    if (infobulleEpinglee && !infobulle.contains(ev.target) && !ev.target.classList.contains('point-frise')) {
      fermerInfobulle();
    }
  });

  // Un bouton "Afficher le tableau détaillé" par ville, juste sous sa carte/frise.
  document.querySelectorAll('.action-ville').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const tableau = document.getElementById('tableau-' + bouton.dataset.ville);
      const visible = tableau.classList.toggle('visible');
      bouton.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
    });
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__BLOCS_CARTE__", blocs_carte)
    .replace("__POINTS__", json.dumps(points, ensure_ascii=False))
    .replace("__POINTS_EDITEURS__", json.dumps(points_editeurs, ensure_ascii=False))
    .replace("__POINTS_GRAVEURS__", json.dumps(points_graveurs, ensure_ascii=False))
    .replace("__VILLES_COORDS__", json.dumps(VILLES_COORDS, ensure_ascii=False))
    .replace("__LIBELLES__", json.dumps(LIBELLES_TECHNIQUE, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Carte + frise écrites dans", CHEMIN_SORTIE)

Carte + frise écrites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/nuage_editions_venise_paris_lyon.html
